In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
import statistics as stats
from scipy import stats as st
import geopandas as gpd


DATA LOADING

In [3]:
Country = pd.read_csv('country_latest.csv')
Health_indicators = pd.read_csv('health_indicators.csv')
indicator_data = pd.read_csv('indicator_metadata.csv')


In [4]:
Country.head()

,country_code,country_name,region,income_level,indicator_code,indicator_name,value,year
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,SP.POP.TOTL,"Population, total",42647492.0,2024
1,ALB,Albania,Europe & Central Asia,Upper middle income,SP.POP.TOTL,"Population, total",2377128.0,2024
2,DZA,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,SP.POP.TOTL,"Population, total",46814308.0,2024
3,ASM,American Samoa,East Asia & Pacific,High income,SP.POP.TOTL,"Population, total",46765.0,2024
4,AND,Andorra,Europe & Central Asia,High income,SP.POP.TOTL,"Population, total",81938.0,2024


In [5]:
Health_indicators.head()

,country_code,country_name,region,income_level,year,population,population_growth,urban_pct,life_expectancy,life_expectancy_male,...,stunting_pct,wasting_pct,overweight_children_pct,undernourishment_pct,fertility_rate,adolescent_fertility,prenatal_care_pct,gdp_per_capita,gdp_per_capita_ppp,poverty_rate
0,ABW,Aruba,Latin America & Caribbean,High income,2000,90588.0,1.030817,65.354550,72.939,70.149,...,NaN,NaN,NaN,NaN,1.845,43.729,NaN,20681.023027,30245.706974,NaN
1,ABW,Aruba,Latin America & Caribbean,High income,2001,91439.0,0.935033,65.335114,73.044,70.251,...,NaN,NaN,NaN,NaN,1.813,40.463,NaN,20740.132583,31920.239073,NaN
2,ABW,Aruba,Latin America & Caribbean,High income,2002,92074.0,0.692052,65.282069,73.135,70.364,...,NaN,NaN,NaN,NaN,1.800,38.185,NaN,21307.248251,31888.508737,NaN
3,ABW,Aruba,Latin America & Caribbean,High income,2003,93128.0,1.138229,65.198292,73.236,70.488,...,NaN,NaN,NaN,NaN,1.808,37.807,NaN,21949.485996,32507.084315,NaN
4,ABW,Aruba,Latin America & Caribbean,High income,2004,95138.0,2.135358,65.088653,73.223,70.487,...,NaN,NaN,NaN,NaN,1.819,38.761,NaN,23700.631990,35059.273098,NaN


In [6]:
indicator_data.head()

,indicator_code,indicator_name,category,unit,description
0,SP.POP.TOTL,"Population, total",Demographics,count,"Population, total (Demographics)"
1,SP.POP.GROW,Population growth (annual %),Demographics,%,Population growth (annual %) (Demographics)
2,SP.URB.TOTL.IN.ZS,Urban population (% of total),Demographics,%,Urban population (% of total) (Demographics)
3,SP.DYN.LE00.IN,"Life expectancy at birth, total",Demographics,years,"Life expectancy at birth, total (Demographics)"
4,SP.DYN.LE00.MA.IN,"Life expectancy, male",Demographics,years,"Life expectancy, male (Demographics)"


In [7]:
Country.info()

<class 'pandas.DataFrame'>
RangeIndex: 7802 entries, 0 to 7801
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    7802 non-null   str    
 1   country_name    7802 non-null   str    
 2   region          7802 non-null   str    
 3   income_level    7802 non-null   str    
 4   indicator_code  7802 non-null   str    
 5   indicator_name  7802 non-null   str    
 6   value           7802 non-null   float64
 7   year            7802 non-null   int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 1.2 MB


In [8]:
Health_indicators.info()

<class 'pandas.DataFrame'>
RangeIndex: 5275 entries, 0 to 5274
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country_code                   5275 non-null   str    
 1   country_name                   5275 non-null   str    
 2   region                         5275 non-null   str    
 3   income_level                   5275 non-null   str    
 4   year                           5275 non-null   int64  
 5   population                     5275 non-null   float64
 6   population_growth              5274 non-null   float64
 7   urban_pct                      5275 non-null   float64
 8   life_expectancy                5275 non-null   float64
 9   life_expectancy_male           5275 non-null   float64
 10  life_expectancy_female         5275 non-null   float64
 11  death_rate                     5275 non-null   float64
 12  birth_rate                     5275 non-null   float64
 13 

In [9]:
Health_indicators.describe()

,year,population,population_growth,urban_pct,life_expectancy,life_expectancy_male,life_expectancy_female,death_rate,birth_rate,health_expenditure_pct_gdp,...,stunting_pct,wasting_pct,overweight_children_pct,undernourishment_pct,fertility_rate,adolescent_fertility,prenatal_care_pct,gdp_per_capita,gdp_per_capita_ppp,poverty_rate
count,5275.000000,5.275000e+03,5274.000000,5275.000000,5275.000000,5275.000000,5275.000000,5275.000000,5275.000000,4514.000000,...,844.000000,843.000000,806.000000,3762.000000,5275.000000,5275.000000,741.000000,5110.000000,4838.000000,1827.000000
mean,2012.000000,3.378305e+07,1.280475,58.716732,70.748244,68.215760,73.394403,8.192702,21.095707,6.222646,...,24.009360,5.896679,6.198015,10.937400,2.797145,51.097317,86.791826,15749.091058,19781.847712,9.492118
std,7.211786,1.323408e+08,1.618914,23.674753,8.701547,8.471691,9.066795,3.412191,10.888048,2.867900,...,15.043393,4.401812,4.499476,11.043652,1.468998,42.893201,16.137098,25074.200336,22391.204701,17.142033
min,2000.000000,9.544000e+03,-11.356645,8.043814,14.665000,12.383000,20.006000,0.841000,4.200000,1.222602,...,0.500000,0.100000,0.000000,2.500000,0.721000,0.505000,16.100000,109.593814,403.983197,0.000000
25%,2006.000000,7.743870e+05,0.297465,39.060949,65.356000,62.722000,67.976000,6.033500,11.746000,4.126287,...,10.800000,2.400000,2.725000,2.500000,1.670000,15.427500,82.500000,1544.567936,3899.860735,0.300000
50%,2012.000000,5.943366e+06,1.175030,60.031069,72.442000,69.217000,75.689000,7.607000,18.172000,5.640688,...,23.800000,4.900000,5.300000,6.300000,2.274000,39.131000,92.800000,5356.365656,11309.514697,1.500000
75%,2018.000000,2.201484e+07,2.210221,77.627285,77.197000,74.500000,80.115000,9.780500,28.806500,7.966407,...,35.325000,8.700000,8.700000,15.600000,3.638500,75.234000,97.600000,20032.321089,28679.803239,9.800000
max,2024.000000,1.450936e+09,21.700343,100.000000,86.497000,84.560000,88.626000,72.483000,53.390000,27.089685,...,64.000000,24.600000,29.600000,72.500000,7.829000,206.357000,100.000000,288001.433369,180939.439450,94.900000


DATA CLEANING